In [6]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


# Model Training: Cascaded Two-Stage Architecture

This notebook trains the cascaded Stage 1 and Stage 2 models for the Early Warning System.

## Architecture
- **Stage 1 (High-Recall Filter)**: Uses high-frequency, continuously available vital signs and readily available demographics to flag at-risk patients early. Designed to maximize recall and generate a dynamic risk score.
- **Stage 2 (High-Precision Classifier)**: For patients flagged by Stage 1, incorporates slower-turnaround laboratory values (e.g., Creatinine, BUN) and comprehensive data to confirm risk. Designed to maximize precision and reduce alarm fatigue.

In [8]:
import os
import sys
import pandas as pd

!pip install optuna

sys.path.append(os.path.abspath('..'))
from src import models, features

PROCESSED_DATA_DIR = '../data/processed'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Load the chronological splits produced by notebook 04 (features + labels together)
print("Loading train / val splits...")
train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'val.parquet'))
print(f"train: {train_df.shape}  |  val: {val_df.shape}")

# Feature sets: Stage 1 = curated vitals/demographics; Stage 2 = all engineered features
stage1_feats = features.get_stage1_features()
stage2_feats = features.get_all_features(train_df)
print(f"Stage-1 features: {len(stage1_feats)}  |  Stage-2 features: {len(stage2_feats)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 11.3 MB/s eta 0:00:00
Loading train / val splits...
train: (1673348, 208)  |  val: (366823, 208)
Stage-1 features: 15  |  Stage-2 features: 197


In [9]:
print("Training Stage 1 models (vitals-only screener, one per horizon)...")
stage1_models = models.train_stage1_models(
    train_df, val_df, feature_cols=stage1_feats, horizons=[6, 12, 24]
)
print("Stage 1 trained for horizons:", list(stage1_models.keys()))

Training Stage 1 models (vitals-only screener, one per horizon)...
Stage 1 trained for horizons: [6, 12, 24]


In [10]:
print("Training Stage 2 models (LightGBM + XGBoost, Optuna-tuned)...")
# FIRST PASS on full MIMIC-IV: n_optuna_trials=15 to validate the whole chain
# end-to-end quickly. Once 01->07 is confirmed working, bump to 50 for the final
# best-tuned models and re-run this notebook.
stage2_models = models.train_stage2_models(
    train_df, val_df, feature_cols=stage2_feats, horizons=[6, 12, 24], n_optuna_trials=15
)
print("Stage 2 trained for horizons:", list(stage2_models.keys()))

Training Stage 2 models (LightGBM + XGBoost, Optuna-tuned)...
Stage 2 trained for horizons: [6, 12, 24]


In [11]:
print("Saving all models to disk...")
models.save_models(stage1_models, stage2_models, MODELS_DIR)
print("Models saved successfully.")

Saving all models to disk...
Models saved successfully.


In [12]:
# Conformal abstention threshold — the COMPOSER-style "I don't know" safety layer.
# Split-conformal (LAC): fit a threshold on the VALIDATION split using the SAME raw
# probabilities the clinical alerts use, for the Stage-2 LightGBM @ 24h model that
# notebook 07 explains. Scores inside the band [1-qhat, qhat] are abstained on.
from src.conformal import fit_conformal_threshold, indeterminate_rate

CONFORMAL_ALPHA = 0.10  # 90% coverage; lower alpha => wider band => abstains more often
conformal_qhat = None

if 24 in stage2_models:
    _m = stage2_models[24]["lgbm"]["model"]
    _p_val = _m.predict_proba(val_df[stage2_feats])[:, 1]
    _y_val = val_df["aki_label_24h"].values
    conformal_qhat = fit_conformal_threshold(_y_val, _p_val, alpha=CONFORMAL_ALPHA)
    lo, hi = 1 - conformal_qhat, conformal_qhat
    rate = indeterminate_rate(_p_val, conformal_qhat)
    print(f"Conformal qhat={conformal_qhat:.3f}  band=[{lo:.3f}, {hi:.3f}]  "
          f"val abstention rate={rate:.1%}")
    if lo > hi:
        print("Note: band empty -> the model is confident enough to never abstain at "
              "this alpha. Expected on the 100-patient demo (overconfident). Lower "
              "CONFORMAL_ALPHA to force a wider gray zone for the demo if you want one.")
else:
    print("No 24h Stage-2 model trained; skipping conformal fit.")

Conformal qhat=0.630  band=[0.370, 0.630]  val abstention rate=17.6%


In [13]:
# Save a self-describing metadata sidecar so the models can be loaded correctly
# on any machine (PC / Drive / HF). Records the trained feature lists, the exact
# library versions, the git commit of the training code, the alert threshold, and
# the conformal abstention threshold. Stdlib only — no new dependencies.
import json
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def _git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return None  # not a git checkout (e.g. fresh Colab download)


metadata = {
    "horizons": [6, 12, 24],
    "alert_threshold": 0.35,  # mirrors src.explain._DEFAULT_THRESHOLD
    "conformal_alpha": CONFORMAL_ALPHA,
    "conformal_qhat_stage2_24h": conformal_qhat,  # None if band/horizon unavailable
    "stage1_features": stage1_feats,
    "stage2_features": stage2_feats,
    "library_versions": {
        p: _ver(p)
        for p in ["lightgbm", "xgboost", "scikit-learn", "numpy", "pandas", "shap"]
    },
    "python": sys.version.split()[0],
    "git_commit": _git_commit(),
}

meta_path = Path(MODELS_DIR) / "model_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))
print(f"Wrote {meta_path}")
print(json.dumps(metadata, indent=2))

Wrote ../models/model_metadata.json
{
  "horizons": [
    6,
    12,
    24
  ],
  "alert_threshold": 0.35,
  "conformal_alpha": 0.1,
  "conformal_qhat_stage2_24h": 0.6304521501093803,
  "stage1_features": [
    "HR",
    "MAP",
    "O2Sat",
    "Temp",
    "Resp",
    "SBP",
    "HR_mean_6h",
    "MAP_mean_6h",
    "MAP_std_6h",
    "HR_std_6h",
    "SBP_mean_6h",
    "Resp_mean_6h",
    "hours_since_sepsis",
    "Age",
    "Gender"
  ],
  "stage2_features": [
    "AST",
    "AST_observed",
    "Age",
    "Alkalinephos",
    "Alkalinephos_observed",
    "BUN",
    "BUN_max_12h",
    "BUN_max_6h",
    "BUN_mean_12h",
    "BUN_mean_6h",
    "BUN_min_12h",
    "BUN_min_6h",
    "BUN_observed",
    "BUN_std_12h",
    "BUN_std_6h",
    "BaseExcess",
    "BaseExcess_observed",
    "Bilirubin_total",
    "Bilirubin_total_observed",
    "Calcium",
    "Calcium_observed",
    "Chloride",
    "Chloride_observed",
    "Creatinine",
    "Creatinine_max_12h",
    "Creatinine_max_6h",
    "Creatini

In [14]:
!git config --global user.email "daemmyoff1cial@gmail.com"
!git config --global user.name "Sng43"

In [ ]:
%cd /content/drive/MyDrive/sentinel-poc
!git add .
!git commit -m "training"
!git push

/content/drive/MyDrive/sentinel-poc
[main 726b291] training
 21 files changed, 9 insertions(+), 502 deletions(-)
 rewrite project_sentinel/models/calibrators/stage1_lgbm_12h_calibrator.pkl (65%)
 rewrite project_sentinel/models/calibrators/stage1_lgbm_24h_calibrator.pkl (69%)
 rewrite project_sentinel/models/calibrators/stage1_lgbm_6h_calibrator.pkl (66%)
 rewrite project_sentinel/models/calibrators/stage2_lgbm_12h_calibrator.pkl (82%)
 rewrite project_sentinel/models/calibrators/stage2_lgbm_24h_calibrator.pkl (83%)
 rewrite project_sentinel/models/calibrators/stage2_lgbm_6h_calibrator.pkl (77%)
 rewrite project_sentinel/models/calibrators/stage2_xgb_12h_calibrator.pkl (84%)
 rewrite project_sentinel/models/calibrators/stage2_xgb_24h_calibrator.pkl (75%)
 rewrite project_sentinel/models/calibrators/stage2_xgb_6h_calibrator.pkl (74%)
 rewrite project_sentinel/models/stage1_lgbm_12h.pkl (81%)
 rewrite project_sentinel/models/stage1_lgbm_24h.pkl (86%)
 rewrite project_sentinel/models/stag

## Training Summary

We have successfully trained the cascaded models:
- **Stage 1 Models** are built on continuously available features (Vitals + Demographics).
- **Stage 2 Models** utilize the full feature set including laboratory results, tuned using Optuna.
- All models are persisted in the `models/` directory for evaluation and subsequent deployment.